# Vélo Demain : exploration et évaluation
Notebook à exécuter après `python -m src.pipeline`. Les sorties sont volontairement vierges ; les résultats calculés sont disponibles dans `reports/`.
Question : prévoir les locations de demain avec les données connues la veille.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'data/raw/day.csv').exists():
    ROOT = ROOT.parent
raw = pd.read_csv(ROOT / 'data/raw/day.csv', parse_dates=['dteday'])
raw.head()


## Vérifier les données
Contrôler les dates manquantes, les doublons et l’identité `casual + registered = cnt`. Pourquoi les deux composantes ne doivent-elles pas servir de variables explicatives ?


In [ ]:
print('Dimensions :', raw.shape)
print('Valeurs manquantes :', raw.isna().sum().sum())
print('Dates dupliquées :', raw.dteday.duplicated().sum())
assert raw.cnt.eq(raw.casual + raw.registered).all()
raw.set_index('dteday').cnt.plot(figsize=(12, 4), title='Demande quotidienne historique')
plt.ylabel('Locations / jour');


## Lire le protocole avant le score
Le test porte sur octobre–décembre 2012. Les retards de test utilisent les vraies observations antérieures : c’est une prévision glissante à J+1. Aucune météo future réalisée n’est utilisée.


In [ ]:
report = json.loads((ROOT / 'reports/metrics.json').read_text(encoding='utf-8'))
pd.DataFrame(report['splits']).T


In [ ]:
pd.DataFrame(report['validation']).T


## Examiner la généralisation
Le modèle est choisi sur validation. Le score du test ne doit pas servir à modifier les paramètres : toute nouvelle itération nécessiterait une nouvelle période de test.


In [ ]:
print('Modèle retenu :', report['selected'])
print('MAE test :', report['test']['mae'])
print('MAE référence :', report['test_baseline']['mae'])
pred = pd.read_csv(ROOT / 'reports/predictions.csv', parse_dates=['dteday'])
pred.set_index('dteday')[['cnt', 'prediction', 'lag_7']].plot(figsize=(12, 4))
plt.ylabel('Locations / jour');


In [ ]:
pred.nlargest(10, 'absolute_error')


## Expérience personnelle à ajouter
1. Formuler une hypothèse sur les erreurs, puis choisir une analyse sur apprentissage et validation.
2. Ajouter une validation à origines glissantes.
3. Comparer les MAE par période et documenter les limites.
4. Ne pas présenter une corrélation comme une cause ni une métrique comme un gain financier réel.

Source : Fanaee-T (2013), UCI Bike Sharing, https://doi.org/10.24432/C5W894, CC BY 4.0.
